# Tutorial 2: Spots

`Spots` represent ray intersection positions on the focal plane — the raw output
of a ray-trace.  Each `Spots` object holds offsets `(dx, dy)` for many rays at
one or more field points, plus a vignetting mask.

Like `FieldCoords`, `Spots` support **frame** conversions (OCS, CCS, DVCS, EDCS)
and **space** conversions (angle ↔ focal_plane).  They also carry a `FieldCoords`
attribute identifying _where_ the spots were measured.

The bridge from `Spots` to `Moments` is `spots.moments(order)` — the key
connection between ray data and image-quality metrics.

In [ ]:
import batoid
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import Angle
import galsim

from StarSharp import FieldCoords, Spots

%matplotlib inline

## 1. Construction

In practice, `Spots` are produced by `RaytracedOpticalModel.spots()`.  Here we
construct them manually to show the structure.

The key arrays have shape `(nfield, nray)`:  
- `dx`, `dy` — ray displacements from the centroid  
- `vignetted` — boolean mask for vignetted rays

In [ ]:
rtp = Angle(30, unit=u.deg)

# Start with a hexapolar grid on the pupil.  Same for each spot.
px, py = batoid.utils.hexapolar(outer=4.18, inner=4.18*0.612, nrad=10)
n_field, n_ray = 3, len(px)

# Simulate spot data: random Zernike perturbations
rng = np.random.default_rng(57721)
coefs = rng.normal(size=(n_field, 12))
dx = np.empty((n_field, n_ray)) * u.um
dy = np.empty((n_field, n_ray)) * u.um
for i in range(n_field):
    zk = galsim.zernike.Zernike(coefs[i], R_outer=4.18, R_inner=4.18*0.612)
    dx[i] = zk.gradX(px, py) * u.um
    dy[i] = zk.gradY(px, py) * u.um

# Mark one edge as vignetted
vignetted = np.repeat(px > 3.5, n_field).reshape(n_field, n_ray)

# Field positions where these spots were computed; made up.
field = FieldCoords(
    x=np.array([0.5, -0.3, 1.0]) * u.deg,
    y=np.array([0.2, 0.8, -0.6]) * u.deg,
    frame="ocs",
    rtp=rtp,
)

spots = Spots(
    dx=dx, dy=dy,
    vignetted=vignetted,
    field=field,
    wavelength=622.0 * u.nm,
    frame="ccs",
    rtp=rtp,
)

print(f"nfield: {spots.nfield}")
print(f"nray:   {spots.nray}")
print(f"frame:  {spots.frame}")
print(f"space:  {spots.space}")
print(f"vignetted count: {spots.vignetted.sum()} / {spots.vignetted.size}")

In [ ]:
# Scalar dx/dy are auto-promoted to 2D
spots_single = Spots(
    dx=rng.normal(size=50) * u.um,
    dy=rng.normal(size=50) * u.um,
    vignetted=np.zeros(50, dtype=bool),
    field=FieldCoords(x=0.5 * u.deg, y=-0.3 * u.deg, frame="ocs", rtp=rtp),
    frame="ccs",
    rtp=rtp,
)
print(f"Auto-promoted shape: dx.shape = {spots_single.dx.shape}")
print(f"nfield={spots_single.nfield}, nray={spots_single.nray}")

### Visualizing a spot diagram

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

for i, ax in enumerate(axes):
    good = ~spots.vignetted[i]
    ax.scatter(
        spots.dx[i, good].to_value(u.um),
        spots.dy[i, good].to_value(u.um),
        s=3, alpha=0.5, label="good",
    )
    ax.scatter(
        spots.dx[i, ~good].to_value(u.um),
        spots.dy[i, ~good].to_value(u.um),
        s=10, c="red", marker="x", label="vignetted",
    )
    ax.set_xlabel("dx [µm]")
    ax.set_ylabel("dy [µm]")
    fx = spots.field.x[i].to_value(u.deg)
    fy = spots.field.y[i].to_value(u.deg)
    ax.set_title(f"Field {i}: ({fx:.1f}°, {fy:.1f}°)")
    ax.set_aspect("equal")
    ax.legend(fontsize=8)

fig.suptitle(f"Spot diagrams ({spots.frame.upper()} frame, {spots.space} space)", fontsize=13)

## 2. Frame conversions

Spots support the same four frames as `FieldCoords`.  The `dx`/`dy` arrays
are rotated (OCS↔CCS), relabeled (CCS↔EDCS), or transposed (CCS↔DVCS).

In [ ]:
spots_ocs = spots.ocs
spots_dvcs = spots.dvcs

print(f"OCS frame:  {spots_ocs.frame}")
print(f"DVCS frame: {spots_dvcs.frame}")

# DVCS swaps x↔y relative to CCS
print(f"\nDVCS.dx == CCS.dy: {np.allclose(spots_dvcs.dx, spots.dy)}")
print(f"DVCS.dy == CCS.dx: {np.allclose(spots_dvcs.dy, spots.dx)}")

In [ ]:
# Roundtrip: CCS → OCS → CCS
roundtrip = spots.ocs.ccs
print(f"dx roundtrip close: {np.allclose(spots.dx, roundtrip.dx)}")
print(f"Max error: {np.max(np.abs(spots.dx - roundtrip.dx)):.2e}")

In [ ]:
# The rotation preserves the vector magnitude
r_ccs = np.sqrt(spots.dx**2 + spots.dy**2)
r_ocs = np.sqrt(spots_ocs.dx**2 + spots_ocs.dy**2)
print(f"Magnitudes preserved: {np.allclose(r_ccs, r_ocs)}")

### Comparing frames visually

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)

# Show field 0 in all four frames
i = 0
good = ~spots.vignetted[i]

for ax, (label, sp) in zip(axes, [
    ("OCS", spots.ocs),
    ("CCS", spots.ccs),
    ("EDCS", spots.edcs),
    ("DVCS", spots.dvcs),
]):
    ax.scatter(
        sp.dx[i, good].to_value(u.um),
        sp.dy[i, good].to_value(u.um),
        s=3, alpha=0.5,
    )
    ax.set_xlabel(f"{label} dx [µm]")
    ax.set_ylabel(f"{label} dy [µm]")
    ax.set_title(label)
    ax.set_aspect("equal")
    ax.set_xlim(-15, 15)
    ax.set_ylim(-15, 15)

fig.suptitle(f"Field 0 spot diagram in all frames (rtp = {rtp.deg:.0f}°)", fontsize=13)

## 3. Space conversions  (angle ↔ focal_plane)

Like `FieldCoords`, `Spots` can be converted between angle and focal-plane space
when a **camera** is attached.  The camera's transforms map the absolute ray
positions (field + offset) through the coordinate transformation, then subtract
the field centroid in the target space.

In [ ]:
# Load camera geometry
from lsst.obs.lsst import LsstCam
camera = LsstCam().getCamera()

# Rebuild field and spots with camera attached
field_cam = FieldCoords(
    x=field.x, y=field.y, frame="ocs", rtp=rtp, camera=camera,
)
spots_cam = Spots(
    dx=dx, dy=dy, vignetted=vignetted, field=field_cam,
    wavelength=622 * u.nm, frame="ccs", rtp=rtp, camera=camera,
)

print(f"Original space: {spots_cam.space}  (units: {spots_cam.dx.unit})")

In [ ]:
# Convert to angle space
spots_ang = spots_cam.angle
print(f"Angle space: {spots_ang.space}  (units: {spots_ang.dx.unit})")
print(f"Frame preserved: {spots_ang.frame}")

In [ ]:
# Roundtrip: focal_plane → angle → focal_plane
spots_rt = spots_cam.angle.focal_plane
print(f"dx roundtrip close: {np.allclose(spots_cam.dx, spots_rt.dx, atol=1e-6)}")
print(f"Max error: {np.max(np.abs(spots_cam.dx - spots_rt.dx)):.2e}")

## 4. Bridge: Spots → Moments

The `compute_moments(order)` method computes 2D image moments from the ray
distribution, excluding vignetted rays.  This is the primary connection between
ray-trace data and image-quality metrics.

Moments are computed in the spot's current frame and units of `µm^order`.

In [ ]:
# Second-order moments
m2 = spots.moments(order=2)
print(f"Type: {type(m2).__name__}")
print(f"Frame: {m2.frame}")
print(f"xx = {m2.xx}")
print(f"xy = {m2.xy}")
print(f"yy = {m2.yy}")

In [ ]:
# Higher-order moments
m3 = spots.moments(order=3)
m4 = spots.moments(order=4)
print(f"Order 3 fields: xxx={m3.xxx[0]:.4f}, xxy={m3.xxy[0]:.4f}, xyy={m3.xyy[0]:.4f}, yyy={m3.yyy[0]:.4f}")
print(f"Order 4 fields: xxxx={m4.xxxx[0]:.2f}, ... yyyy={m4.yyyy[0]:.2f}")

In [ ]:
# Frame consistency: computing moments commutes with frame conversion
m2_ocs_from_spots = spots.ocs.moments(order=2)
m2_ocs_from_moments = spots.moments(order=2).ocs

print("Moments from OCS spots vs. CCS moments rotated to OCS:")
print(f"  xx close: {np.allclose(m2_ocs_from_spots.xx, m2_ocs_from_moments.xx)}")
print(f"  xy close: {np.allclose(m2_ocs_from_spots.xy, m2_ocs_from_moments.xy)}")
print(f"  yy close: {np.allclose(m2_ocs_from_spots.yy, m2_ocs_from_moments.yy)}")

## 5. Indexing and batch dimensions

In [ ]:
# Integer indexing selects a field point
sp0 = spots[0]
print(f"spots[0]: nfield={sp0.nfield}, nray={sp0.nray}")
print(f"  dx shape: {sp0.dx.shape}")

# Slice indexing
sp_first2 = spots[:2]
print(f"spots[:2]: nfield={sp_first2.nfield}")

In [ ]:
print(f"batch_shape: {spots.batch_shape}")
print(f"nfield: {spots.nfield}, nray: {spots.nray}")

## 6. Pupil coordinates

Spots can optionally carry pupil (stop-plane) coordinates `px`, `py` for each
ray.  These are 1D arrays of shape `(nray,)` shared across all field points.  The pupil
coordinates are always in the OCS frame.

In [ ]:
# Simulate pupil coordinates on a hexapolar grid
theta = np.linspace(0, 2*np.pi, n_ray, endpoint=False)
r = rng.uniform(0, 4.18, n_ray)
px = r * np.cos(theta) * u.m
py = r * np.sin(theta) * u.m

spots_with_pupil = Spots(
    dx=dx, dy=dy, vignetted=vignetted, field=field,
    wavelength=622 * u.nm, frame="ccs", rtp=rtp,
    px=px, py=py,
)
print(f"Pupil coordinates: px.shape={spots_with_pupil.px.shape}")
print(f"Pupil coords are preserved through frame conversion: "
      f"{spots_with_pupil.ocs.px is not None}")

## 7. Summary

| Property / Method | Description |
|---|---|
| `dx`, `dy` | Ray offsets, shape `(nfield, nray)` |
| `vignetted` | Boolean vignetting mask |
| `.field` | `FieldCoords` where spots were computed |
| `.frame`, `.space` | Current frame and space |
| `.ocs`, `.ccs`, `.dvcs`, `.edcs` | Frame conversion |
| `.angle`, `.focal_plane` | Space conversion (requires `camera`) |
| `.moments(order)` | **Bridge to Moments** |
| `.nfield`, `.nray`, `.batch_shape` | Shape info |
| `px`, `py` | Optional pupil coordinates |

**Next:** [03_Moments.ipynb](03_Moments.ipynb) — image moments and their
frame-dependent algebra.